<a href="https://colab.research.google.com/github/davesagit123/blank-app/blob/main/fence_new_apodex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import statistics as st
import pandas as pd
import io

# --- Paste your raw data here ---
raw_data_string = """
1	Barrelling	-6.31L	52	-1.82L	-1.53L	-1.12L	-0.29L
2	Fagin	+5.32L	51	-0.20L	-0.10L	+0.31L	+0.44L
3	Sir Beveridge	-2.26L	50	-2.05L	-1.73L	-0.90L	-0.09L
4	Baccarat Rouge	-1.42L	49	-1.12L	-1.24L	-0.76L	-0.03L
5	Butterfly Style	-1.15L	48	-0.75L	-0.61L	-0.24L	+0.08L
6	Royal Gladiator	+1.70L	44	-0.26L	-0.38L	-0.10L	+0.31L
7	Financially Famous	+2.27L	28	-0.64L	-0.55L	-0.37L	-0.02L
8	Capital Chord	-1.71L	42	-0.91L	-0.77L	-0.67L	-0.09L
9	Silver Smash	+3.96L	24	-0.69L	-1.12L	-0.99L	-0.31L
10	Poutchek	-2.80L	50	-1.05L	-1.06L	-0.41L	+0.19L
"""

# --- Parsing the raw string into the runners list ---
runners = [tuple(line.split('\t')) for line in raw_data_string.strip().split('\n')]

# --- Column names (in calculation order) ---
COLUMNS = ["Runner Time", "Early Pace", "L800m", "L600m", "L400m", "L200m"]

# --- Parse: strip 'L' and '+' signs; convert to float ---
def parse(x):
    s = str(x).strip()
    if s.endswith('L'):
        s = s[:-1]
    return float(s.replace('+',''))

# Build per-column arrays from the table (columns 3..6 of each row)
data = {}
for i, col in enumerate(COLUMNS):
    data[col] = [parse(r[2+i]) for r in runners]

def quartile_exc(sorted_vals, p):
    """Exclusive percentile, QUARTILE.EXC style: position = p*(n+1), interpolated."""
    n = len(sorted_vals)
    pos = p * (n + 1)
    j = int(pos); f = pos - j
    if j < 1:  return sorted_vals[0]
    if j >= n: return sorted_vals[-1]
    return sorted_vals[j-1] + f * (sorted_vals[j] - sorted_vals[j-1])

def fence_stats(vals):
    sv = sorted(vals)
    q1  = quartile_exc(sv, 0.25)
    q3  = quartile_exc(sv, 0.75)
    iqr = q3 - q1
    lo  = q1 - 1.5*iqr
    hi  = q3 + 1.5*iqr
    mu  = sum(vals)/len(vals)
    sd_pop = st.pstdev(vals)
    sd_sam = st.stdev(vals)
    new_fence_s  = lo + sd_sam
    new_fence_sig= lo + sd_pop
    return dict(Q1=q1, Q3=q3, IQR=iqr, LOWER=lo, UPPER=hi,
                MEAN=mu, SD_POP=sd_pop, SD_SAM=sd_sam,
                NEWFENCE_S=new_fence_s, NEWFENCE_SIG=new_fence_sig)

RESULTS = {col: fence_stats(data[col]) for col in COLUMNS}

fmt = lambda v: f"{v:+.4f}"
rows = []
metrics = ["MEAN", "SD_POP", "SD_SAM", "Q1", "Q3", "IQR", "LOWER", "UPPER", "NEWFENCE_S"]
metric_labels = ["Mean", "StdDev σ (pop)", "StdDev s (sample)", "Q1", "Q3", "IQR", "Lower Fence", "Upper Fence", "**New Fence (LF + s)**"]

for label, key in zip(metric_labels, metrics):
    rows.append([label] + [RESULTS[col][key] for col in COLUMNS])

df = pd.DataFrame(rows, columns=["Metric"] + COLUMNS).set_index("Metric")
display(df.style.format(fmt))

rt = RESULTS.get("Runner Time")
if rt:
    print(f"\nRunner Time Validation:")
    print(f"Lower fence   : {rt['LOWER']:.5f}")
    print(f"Upper fence   : {rt['UPPER']:.5f}")
    print(f"s (sample std): {rt['SD_SAM']:.13f}")
    print(f"New fence     : {rt['NEWFENCE_S']:.13f}")

,Runner Time,Early Pace,L800m,L600m,L400m,L200m
Metric,,,,,,
Mean,-0.2400,+43.8000,-0.9490,-0.9090,-0.5250,+0.0190
StdDev σ (pop),+3.3238,+9.4106,+0.5701,+0.4908,+0.4223,+0.2294
StdDev s (sample),+3.5036,+9.9197,+0.6009,+0.5173,+0.4452,+0.2418
Q1,-2.3950,+38.5000,-1.2950,-1.3125,-0.9225,-0.1400
Q3,+2.6925,+50.2500,-0.5450,-0.5075,-0.2050,+0.2200
IQR,+5.0875,+11.7500,+0.7500,+0.8050,+0.7175,+0.3600
Lower Fence,-10.0262,+20.8750,-2.4200,-2.5200,-1.9987,-0.6800
Upper Fence,+10.3238,+67.8750,+0.5800,+0.7000,+0.8712,+0.7600
**New Fence (LF + s)**,-6.5226,+30.7947,-1.8191,-2.0027,-1.5536,-0.4382



Runner Time Validation:
Lower fence   : -10.02625
Upper fence   : 10.32375
s (sample std): 3.5036108358219
New fence     : -6.5226391641781


In [ ]:
# --- Runner table (as given) ---
runners = [
    ("1","Barrelling",     "-6.31L", "52",  "-1.82L","-1.53L","-1.12L","-0.29L"),
    ("2","Fagin",          "+5.32L", "51",  "-0.20L","-0.10L","+0.31L","+0.44L"),
    ("3","Sir Beveridge",  "-2.26L", "50",  "-2.05L","-1.73L","-0.90L","-0.09L"),
    ("4","Baccarat Rouge", "-1.42L", "49",  "-1.12L","-1.24L","-0.76L","-0.03L"),
    ("5","Butterfly Style","-1.15L", "48",  "-0.75L","-0.61L","-0.24L","+0.08L"),
    ("6","Royal Gladiator","+1.70L", "44",  "-0.26L","-0.38L","-0.10L","+0.31L"),
    ("7","Financially Famous","+2.27L","28","-0.64L","-0.55L","-0.37L","-0.02L"),
    ("8","Capital Chord",  "-1.71L", "42",  "-0.91L","-0.77L","-0.67L","-0.09L"),
    ("9","Silver Smash",   "+3.96L", "24",  "-0.69L","-1.12L","-0.99L","-0.31L"),
    ("10","Poutchek",      "-2.80L", "50",  "-1.05L","-1.06L","-0.41L","+0.19L"),
]

# --- Column names (in calculation order) ---
COLUMNS = ["Runner Time", "Early Pace", "L800m", "L600m", "L400m", "L200m"]

# --- Parse: strip 'L' and '+' signs; convert to float ---
def parse(x):
    s = str(x).strip()
    if s.endswith('L'):
        s = s[:-1]
    return float(s.replace('+',''))

# Build per-column arrays from the table (columns 3..6 of each row)
data = {}
for i, col in enumerate(COLUMNS):
    data[col] = [parse(r[2+i]) for r in runners]

for col, vals in data.items():
    print(col, vals)

import statistics as st

def quartile_exc(sorted_vals, p):
    """Exclusive percentile, QUARTILE.EXC style: position = p*(n+1), interpolated."""
    n = len(sorted_vals)
    pos = p * (n + 1)
    j = int(pos); f = pos - j
    if j < 1:  return sorted_vals[0]
    if j >= n: return sorted_vals[-1]
    return sorted_vals[j-1] + f * (sorted_vals[j] - sorted_vals[j-1])

def fence_stats(vals):
    sv = sorted(vals)
    q1  = quartile_exc(sv, 0.25)
    q3  = quartile_exc(sv, 0.75)
    iqr = q3 - q1
    lo  = q1 - 1.5*iqr          # Lower fence  (hackmath)
    hi  = q3 + 1.5*iqr          # Upper fence  (hackmath)
    mu  = sum(vals)/len(vals)
    sd_pop = st.pstdev(vals)    # σ  (population std dev, ddof=0)
    sd_sam = st.stdev(vals)     # s  (sample   std dev, ddof=1)
    new_fence_s  = lo + sd_sam  # matches user's verified example: -6.5226391641781
    new_fence_sig= lo + sd_pop  # alternative using σ
    return dict(Q1=q1, Q3=q3, IQR=iqr, LOWER=lo, UPPER=hi,
                MEAN=mu, SD_POP=sd_pop, SD_SAM=sd_sam,
                NEWFENCE_S=new_fence_s, NEWFENCE_SIG=new_fence_sig)

RESULTS = {col: fence_stats(data[col]) for col in COLUMNS}

import pandas as pd

fmt = lambda v: f"{v:+.4f}"
rows = [
    ("Mean",               [r["MEAN"]   for r in RESULTS.values()]),
    ("StdDev σ (pop)",     [r["SD_POP"] for r in RESULTS.values()]),
    ("StdDev s (sample)",  [r["SD_SAM"] for r in RESULTS.values()]),
    ("Q1",                 [r["Q1"]    for r in RESULTS.values()]),
    ("Q3",                 [r["Q3"]    for r in RESULTS.values()]),
    ("IQR",                [r["IQR"]   for r in RESULTS.values()]),
    ("Lower Fence",        [r["LOWER"]  for r in RESULTS.values()]),
    ("Upper Fence",        [r["UPPER"]  for r in RESULTS.values()]),
    ("**New Fence (LF + s)**", [r["NEWFENCE_S"] for r in RESULTS.values()]),
    ("(alt) LF + σ",       [r["NEWFENCE_SIG"] for r in RESULTS.values()]),
]
df = pd.DataFrame(rows, index=COLUMNS).T
display(df.style.format(fmt))


rt = RESULTS["Runner Time"]
print(f"Lower fence   : {rt['LOWER']:.5f}   (expected -10.02625)")
print(f"Upper fence   : {rt['UPPER']:.5f}   (expected  10.32375)")
print(f"σ (pop std)   : {rt['SD_POP']:.13f}   (expected 3.3238170828131)")
print(f"s (sample std): {rt['SD_SAM']:.13f}   (expected 3.5036108358219)")
print(f"New fence     : {rt['NEWFENCE_S']:.13f}   (expected -6.5226391641781)")
assert abs(rt['LOWER']  - (-10.02625))  < 1e-9, "lower fence mismatch"
assert abs(rt['UPPER']  - (10.32375))   < 1e-9, "upper fence mismatch"
assert abs(rt['NEWFENCE_S'] - (-6.5226391641781)) < 1e-9, "new fence mismatch"
print("ALL CHECKS PASSED ✔")
